In [ ]:
import os 
import numpy as np
import pandas as pd
from scipy.stats import zscore

import scanpy as sc 

import matplotlib.pyplot as plt
import seaborn as sns

from scroutines import powerplots

In [ ]:
outfigdir = "/u/home/f/f7xiesnm/project-zipursky/v1-bb/results_v1astro/260114"
!mkdir -p $outfigdir
fig_manager = powerplots.FigManager(outfigdir)

In [ ]:
ddir = "/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/merfish/organized" 
!ls -alhtr $ddir/P28NRDR*

In [ ]:
ddir2 = "/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/astro_john/cellchat_from_riki"
!ls $ddir2

In [ ]:
f = "/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/merfish/merfish_genes.txt"
genes_mer = np.loadtxt(f, dtype=str)
genes_mer.shape

# CellChat data
- forget about `.y` for now

In [ ]:
f1_1 = os.path.join(ddir2, 'Cell-Cell_Contact_p28nr_vs_28dr.csv')
f1_2 = os.path.join(ddir2, 'ECM-Receptor_p28nr_vs_28dr.csv')
f1_3 = os.path.join(ddir2, 'Secreted_Signaling_p28nr_vs_28dr.csv')
df1_1 = pd.read_csv(f1_1, index_col=0)
df1_2 = pd.read_csv(f1_2, index_col=0)
df1_3 = pd.read_csv(f1_3, index_col=0)
df1 = pd.concat([df1_1, df1_2, df1_3], axis=0)
df1

In [ ]:
cond_x = np.logical_xor(
    df1['source.x'] == 'AST', 
    df1['target.x'] == 'AST', 
)
cond_y = np.logical_xor(
    df1['source.y'] == 'AST', 
    df1['target.y'] == 'AST', 
)
cond = np.logical_or(cond_x, cond_y)

print(cond.sum())
df1sub = df1[cond]
df1sub = df1sub.fillna('NA')
df1sub

In [ ]:
df1sub['receptor.x'].unique()

In [ ]:
mols_x = np.union1d(df1sub['ligand.x'].unique(), 
                    df1sub['receptor.x'].unique())

mols_y = np.union1d(df1sub['ligand.y'].unique(), 
                    df1sub['receptor.y'].unique())

mols = np.union1d(mols_x, mols_y)

mols_mer = np.intersect1d(mols, genes_mer)
len(mols), len(mols_mer), mols, mols_mer

In [ ]:
cond2_x = np.logical_and(
    df1sub['ligand.x'].isin(mols_mer),
    df1sub['receptor.x'].isin(mols_mer),
)
cond2_y = np.logical_and(
    df1sub['ligand.y'].isin(mols_mer),
    df1sub['receptor.y'].isin(mols_mer),
)
cond2 = np.logical_or(cond2_x, cond2_y)
df1sub2 = df1sub[cond2]
len(df1sub2)

In [ ]:
df1sub2.sort_values('abslogPdif', ascending=False).to_csv('check_nrdr.csv')

In [ ]:
print(df1sub2['ligand.x'].unique())
print(df1sub2['receptor.x'].unique())
print(df1sub2['ligand.y'].unique())
print(df1sub2['receptor.y'].unique())

genes_cellchat = np.unique(np.hstack([df1sub2['ligand.x'], 
                                      df1sub2['receptor.x'],
                                      df1sub2['ligand.y'], 
                                      df1sub2['receptor.y'],
                                     ]))

genes_cellchat = genes_cellchat[genes_cellchat!="NA"]
genes_cellchat

In [ ]:
df1sub2s_x = df1sub2[df1sub2['source.x']=='AST']
df1sub2t_x = df1sub2[df1sub2['target.x']=='AST']

df1sub2s_y = df1sub2[df1sub2['source.y']=='AST']
df1sub2t_y = df1sub2[df1sub2['target.y']=='AST']

In [ ]:
summary_s_x = df1sub2s_x.groupby(['target.x', 'interaction_name.x']).first()['prob.x'].unstack().fillna(0).sort_index().sort_index(axis=1)
summary_t_x = df1sub2t_x.groupby(['source.x', 'interaction_name.x']).first()['prob.x'].unstack().fillna(0).sort_index().sort_index(axis=1)
summary_s_x = summary_s_x.loc[['L2/3', 'L4', 'L5IT', 'L5PT', 'L6CTb', 'Pvalb', 'Sst', 'Vlp']]
summary_t_x = summary_t_x.loc[['L2/3', 'L4', 'L5IT', 'L5PT', 'L6CTb', 'Pvalb', 'Sst', 'Vlp']]
summary_s_x.shape, summary_t_x.shape

In [ ]:
summary_s_y = df1sub2s_y.groupby(['target.y', 'interaction_name.y']).first()['prob.y'].unstack().fillna(0).sort_index().sort_index(axis=1)
summary_t_y = df1sub2t_y.groupby(['source.y', 'interaction_name.y']).first()['prob.y'].unstack().fillna(0).sort_index().sort_index(axis=1)
summary_s_y = summary_s_y.loc[['L2/3', 'L4', 'L5IT', 'L5PT', 'L6CTb', 'Pvalb', 'Sst', 'Vlp']]
summary_t_y = summary_t_y.loc[['L2/3', 'L4', 'L5IT', 'L5PT', 'L6CTb', 'Pvalb', 'Sst', 'Vlp']]
summary_s_y.shape, summary_t_y.shape

In [ ]:
fig, axs = plt.subplots(1,4,figsize=(2.5*4,4))
ax = axs[0]
sns.heatmap(summary_s_x, 
            cbar_kws={'shrink':0.5, }, cmap='rocket_r', ax=ax)
ax.set_title('Astro -> X')

ax = axs[1]
sns.heatmap(summary_s_y, 
            cbar_kws={'shrink':0.5, }, cmap='rocket_r', ax=ax)
ax.set_title('Astro -> X (DR)')

ax = axs[2]
sns.heatmap(summary_t_x, 
            cbar_kws={'shrink':0.5, }, cmap='rocket_r', ax=ax)
ax.set_title('X -> Astro')

ax = axs[3]
sns.heatmap(summary_t_y, 
            cbar_kws={'shrink':0.5, }, cmap='rocket_r', ax=ax)
ax.set_title('X -> Astro (DR)')

fig.tight_layout()
plt.show()

# RNA data

In [ ]:
from scipy import sparse

In [ ]:
def preprocessing(adata):
    # filter genes
    cond = np.ravel((adata.X>0).sum(axis=0)) > 10 # expressed in more than 10 cells
    adata_sub = adata[:,cond]

    # counts
    x = adata_sub.X
    cov = adata_sub.obs['n_counts'].values

    # CP10k
    xn = (sparse.diags(1/cov).dot(x))*1e4

    # log10(CP10k+1)
    xln = xn.copy()
    xln.data = np.log10(xln.data+1)

    adata_sub.layers['norm'] = xn
    adata_sub.layers['lognorm'] = xln
    
    return adata_sub

In [ ]:
def get_hvgs(adata, layer, nbin=20, qth=0.3):
    """
    """
    xn = adata.layers[layer]
    
    # min
    gm = np.ravel(xn.mean(axis=0))

    # var
    tmp = xn.copy()
    tmp.data = np.power(tmp.data, 2)
    gv = np.ravel(tmp.mean(axis=0))-gm**2

    # cut 
    lbl = pd.qcut(gm, nbin, labels=np.arange(nbin))
    gres = pd.DataFrame()
    gres['lbl'] = lbl
    gres['mean'] = gm
    gres['var'] = gv
    gres['ratio']= gv/gm

    # select
    gres_sel = gres.groupby('lbl')['ratio'].nlargest(int(qth*(len(gm)/nbin))) #.reset_index()
    gsel_idx = np.sort(gres_sel.index.get_level_values(1).values)

    assert np.all(gsel_idx != -1)
    
    return adata.var.index.values[gsel_idx]

In [ ]:
#
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/cheng21_cell_scrna/organized/P28NR.h5ad'
adata_rna_raw = sc.read(f)
adata_rna_raw

In [ ]:
adata_rna_raw = preprocessing(adata_rna_raw)
adata_rna_raw

In [ ]:
genes_cellchat_rna = np.intersect1d(adata_rna_raw.var.index.values, genes_cellchat)
genes_cellchat_rna

# MERFISH data - check these 5 genes

In [ ]:
import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_merfish')
import merfish_genesets
import merfish_datasets
import utils_merfish

from scroutines import basicu

In [ ]:
def unify_labels(a, b):
    """
    """
    b = b.replace('NA', np.nan)
    
    cat_a = a.cat.categories
    cat_b = b.cat.categories
    
    if len(cat_a)==len(cat_b) and np.all(cat_a==cat_b):
        u = a.fillna(b)
    else:
        cats = np.union1d(cat_a, cat_b)
        a = a.cat.set_categories(cats).copy()
        b = b.cat.set_categories(cats).copy()
        u = a.fillna(b)
        
    return u

In [ ]:
%%time
names = [
    'P28NRa_ant', 
    'P28NRa_pos',
    
    'P28NRb_ant', 
    'P28NRb_pos',
    
    'P28DRa_ant', 
    'P28DRa_pos',
    
    'P28DRb_ant', 
    'P28DRb_pos',
]

alldata = {}
for name in names:
    adatasub = sc.read(os.path.join(ddir, f'{name}_l2_v1_240723.h5ad')) 
    adatasub.obs.index = np.char.add(f'{name}', adatasub.obs.index.values)
    alldata[name] = adatasub 
    print(name, len(alldata[name]))
    
genes = adatasub.var.index.values
genes.shape

In [ ]:
genesets, df = merfish_genesets.get_all_genesets()
iegs   = genesets['i']
genes_noniegs = np.array([g for g in genes if g not in iegs])
igenes_idx = basicu.get_index_from_array(adatasub.var.index.values, iegs)

In [ ]:
mean_total_rna_target = 250
adata_merged = []
for i, name in enumerate(names):
    j = i // 4
    i = i % 4
    
    adatasub = alldata[name].copy()
    adatasub.obs['sample'] = name
    
    norm_cnts = adatasub.layers['norm']
    # mean_per_batch = np.mean(norm_cnts.sum(axis=1))
    mean_per_batch_noniegs = np.mean(adatasub[:,genes_noniegs].layers['norm'].sum(axis=1))
    
    adatasub.layers['jnorm']  = norm_cnts*(mean_total_rna_target/mean_per_batch_noniegs)
    adatasub.layers['ljnorm'] = np.log2(1+adatasub.layers['jnorm'])
    
    adatasub.obs['norm_transcript_count']  = adatasub.layers['norm'].sum(axis=1)
    adatasub.obs['jnorm_transcript_count'] = adatasub.layers['jnorm'].sum(axis=1)
    
    adatasub.obs['depth_show'] = -adatasub.obs['depth'].values - i*1300 # name
    adatasub.obs['width_show'] =  adatasub.obs['width'].values - np.min(adatasub.obs['width'].values) + j*2500   # name
    
    adata_merged.append(adatasub)
    
adata_merged = sc.concat(adata_merged)
adata_merged

# Metadata

In [ ]:
f = os.path.join(ddir, "P28NRDR_v1_rna_merfish_240729.h5ad")
adata = sc.read(f)

f = os.path.join(ddir, "P28NRDR_v1glut_rna_merfish_240729.h5ad")
adata_exc = sc.read(f, backed='r')
f = os.path.join(ddir, "P28NRDR_v1gaba_rna_merfish_240729.h5ad")
adata_inh = sc.read(f, backed='r')

# organize uClass and uSubclass
a = adata.obs['Class_broad']
b = adata.obs['gated_pred']
adata.obs['uClass'] = unify_labels(a, b)

a = adata_exc.obs['Subclass']
b = adata_exc.obs['gated_pred_subclass']
adata_exc.obs['uSubclass'] = unify_labels(a, b)

a = adata_inh.obs['Subclass']
b = adata_inh.obs['gated_pred_subclass']
adata_inh.obs['uSubclass'] = unify_labels(a, b)

uSubclass = pd.concat([adata_exc.obs['uSubclass'], adata_inh.obs['uSubclass']])
adata.obs = adata.obs.join(uSubclass)
adata.obs['uSubclass'] = adata.obs['uSubclass'].fillna(adata.obs['uClass'])
adata.obs

In [ ]:
adata_merged.obs

In [ ]:
adata_merged = adata_merged[adata[adata.obs['modality']=='merfish'].obs.index]
adata_merged_nr = adata_merged[adata_merged.obs['sample'].str.contains(r'^P28NR')]
adata_merged_dr = adata_merged[adata_merged.obs['sample'].str.contains(r'^P28DR')]
adata_merged

In [ ]:
adata_merged_nr

In [ ]:
adata_merged_dr

In [ ]:
genes_cellchat_mer = np.intersect1d(genes, genes_cellchat)
genes_cellchat_mer

In [ ]:
genes_show = genes_cellchat_mer #['Cdh4', 'Negr1', 'Ptprm', 'Plxna1', 'Sema6d']

In [ ]:
adata_rna = adata[adata.obs['modality']=='rna']

adata_mer = adata[adata.obs['modality']=='merfish']
adata_mer_nr = adata_mer[adata_mer.obs['sample'].str.contains(r'^P28NR')]
adata_mer_dr = adata_mer[adata_mer.obs['sample'].str.contains(r'^P28DR')]

print(adata_rna.shape)
print(adata_mer.shape)
print(adata_mer_nr.shape)
print(adata_mer_dr.shape)

In [ ]:
adata_rna

In [ ]:
adata_rna.obs['sample'].unique()

In [ ]:
adata_mer.obs['sample'].unique()


In [ ]:
sel_adata_raw = adata_rna_raw
sel_adata = adata_rna
sel_col = 'uSubclass'
sel_layer = 'lognorm'
sel_genes = genes_show
sel_data = np.array(sel_adata_raw[:,sel_genes].layers[sel_layer].todense())

dfval = sel_adata.obs[[sel_col]].copy()
dfval[sel_genes] = sel_data
dfval_mean = dfval.groupby(sel_col).mean()
dfval_mean_zscore = zscore(dfval_mean, axis=0)
print(dfval_mean.mean())

In [ ]:
sel_adata_raw = adata_merged_nr
sel_adata = adata_mer_nr 
sel_col = 'uSubclass'
sel_layer = 'ljnorm'
sel_genes = genes_show
sel_data = np.array(sel_adata_raw[:,sel_genes].layers[sel_layer])

dfval2 = sel_adata.obs[[sel_col]].copy()
dfval2[sel_genes] = sel_data
dfval2_mean = dfval2.groupby(sel_col).mean()
dfval2_mean_zscore = zscore(dfval2_mean, axis=0)
print(dfval2_mean.mean())

In [ ]:
sel_adata_raw = adata_merged_dr
sel_adata = adata_mer_dr 
sel_col = 'uSubclass'
sel_layer = 'ljnorm'
sel_genes = genes_show
sel_data = np.array(sel_adata_raw[:,sel_genes].layers[sel_layer])

dfval3 = sel_adata.obs[[sel_col]].copy()
dfval3[sel_genes] = sel_data
dfval3_mean = dfval3.groupby(sel_col).mean()
dfval3_mean_zscore = zscore(dfval3_mean, axis=0)
print(dfval3_mean.mean())

In [ ]:
selrows = [
    "Astrocytes", 
    
    "L2/3", "L4", 
    "L5IT", "L5PT", 
    "L6CT", "L6IT",
    "Pvalb", "Sst", "Vip",
    
    # "Endothelial",
    # "OPCs",
    # "Oligodendrocytes",
    # "Microglia",
    # "VLMCs",
]

selcols = [
    'Cdh4', 'Negr1', 'Ptprm',
    'Sema6a', 'Sema6d', 'Sdc4', 'Mertk', 
    'Plxna4', 'Plxna1', 'Col6a1', 'Gas6',
    'Megf10',
]

fig, axs = plt.subplots(1,3,figsize=(4*3,1*4))
ax = axs[0]
ax.set_title('snRNA-seq (NR)')
sns.heatmap( dfval_mean_zscore.loc[selrows][selcols],
            vmax=2, vmin=-2, cmap='coolwarm', 
            cbar_kws=dict(shrink=0.5),
            ax=ax)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

ax = axs[1]
ax.set_title('MERFISH (NR)')
sns.heatmap(dfval2_mean_zscore.loc[selrows][selcols], 
            vmax=2, vmin=-2, cmap='coolwarm', 
            cbar_kws=dict(shrink=0.5),
            ax=ax)

ax = axs[2]
ax.set_title('MERFISH (DR)')
sns.heatmap(dfval3_mean_zscore.loc[selrows][selcols], 
            vmax=2, vmin=-2, cmap='coolwarm', 
            cbar_kws=dict(shrink=0.5),
            ax=ax)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
plt.show()

# visualize 1 gene
- Col6a1

In [ ]:
adata_mer.obs.columns

In [ ]:
width_min = adata_mer_nr.obs.groupby('sample')['width'].min().reindex(names)
width_max = adata_mer_nr.obs.groupby('sample')['width'].max().reindex(names)
width_rng = width_max - width_min 
width_cum = pd.Series(np.cumsum(np.hstack([0, width_rng[:-1]+100])), index=names)

adata_mer_nr.obs['width_n0']    =  adata_mer_nr.obs['width']    - width_min.reindex(adata_mer_nr.obs['sample']).values
adata_mer_nr.obs['width_show2'] =  adata_mer_nr.obs['width_n0'] + width_cum.reindex(adata_mer_nr.obs['sample']).values
adata_mer_nr.obs['depth_show2'] = -adata_mer_nr.obs['depth']

In [ ]:


adata_data_raw = adata_merged_nr
adata_meta = adata_mer_nr

gns = [ 
       'Sema6a', 'Sema6d', 'Sdc4', 'Mertk', 
       'Plxna4', 'Plxna1', 'Col6a1', 'Gas6',
       'Cdh4', 'Negr1', 'Ptprm',
      ] 
n = len(gns)
x =  adata_meta.obs['width_show2']
y =  adata_meta.obs['depth_show2']

fig, axs = plt.subplots(n,1,figsize=(1*10,n*1))
for i, (ax, gn) in enumerate(zip(axs, gns)):
    # sample titles
    if i == 0:
        for lbl, coord in width_cum.items():
            ax.text(coord, 100, lbl, fontsize=8)# , pad=20)
    
    g = adata_data_raw[:,gn].layers['ljnorm'].reshape(-1,)
    vmax = np.percentile(g, 99)
    vmin = np.percentile(g,  0)
    sorting = np.argsort(g)
    
    p = utils_merfish.st_scatter_ax(fig, ax,  x[sorting],  y[sorting],  gexp=g[sorting], s=1, title='', vmin=vmin, vmax=vmax, cmap='rocket_r')
    # p = utils_merfish.st_scatter_ax(fig, ax,  x,  y,  gexp=g, s=1, title='', vmin=vmin, vmax=vmax, cmap='rocket_r')
    ax.set_title(gn, loc='left', va='center', ha='right', y=0.5, pad=None)
    fig.colorbar(p, pad=0, shrink=0.5, aspect=5, ticks=[np.round(vmin, decimals=1), np.round(vmax-0.1, decimals=1)])
    
fig_manager.savefig(fig)
plt.show()
    

In [ ]:
adata_data_raw = adata_merged_nr
adata_meta = adata_mer_nr

gns = [ 
       'Sema6a', 'Sema6d', 'Sdc4', 'Mertk', 
       'Plxna4', 'Plxna1', 'Col6a1', 'Gas6',
       'Cdh4', 'Negr1', 'Ptprm',
      ] 
n = len(gns)
x =  adata_meta.obs['umap1']
y =  adata_meta.obs['umap2']

nx = 4
ny = int((n+nx-1)/nx)

fig, axs = plt.subplots(ny,nx,figsize=(3*nx,ny*3))
for i, (ax, gn) in enumerate(zip(axs.flat, gns)):
    
    g = adata_data_raw[:,gn].layers['ljnorm'].reshape(-1,)
    vmax = np.percentile(g, 99)
    vmin = np.percentile(g,  0)
    sorting = np.argsort(g)
    
    # p = utils_merfish.st_scatter_ax(fig, ax,  x[sorting],  y[sorting],  gexp=g[sorting], s=2, title='', vmin=vmin, vmax=vmax, cmap='rocket_r')
    p = utils_merfish.st_scatter_ax(fig, ax,  x,  y,  gexp=g, s=2, title='', vmin=vmin, vmax=vmax, cmap='rocket_r')
    ax.set_title(gn)
    fig.colorbar(p, pad=0, shrink=0.2, aspect=10, ticks=[np.round(vmin, decimals=1), np.round(vmax-0.1, decimals=1)])

for ax in axs.flat[i:]:
    ax.axis('off')
    
fig_manager.savefig(fig)
plt.show()
    

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(8*2,6), sharex=True, sharey=True)
x = 'umap1'
y = 'umap2'
# hue = 'uSubclass'
hue = 'uClass'
for ax, adata_mod in zip(axs, [adata_mer_nr, adata_rna]):
    sns.scatterplot(data=adata_mod.obs.sample(frac=1, replace=False), 
                    x=x, y=y, 
                    hue=hue, 
                    # palette=clsts_palette3, hue_order=list(clsts_palette3.keys()),
                    s=5, edgecolor='none', ax=ax, rasterized=True)
    ax.axis('off')
    ax.set_aspect('equal')
    ax.legend(bbox_to_anchor=(1,1), fontsize=10)

axs[0].set_title('snRNA-seq (Cheng et al. 2022)')
axs[1].set_title('MERFISH (Xie et al. 2025)')
# powerplots.savefig_autodate(fig, os.path.join(outdir, 'fig1_umap_exc.pdf'))
fig.tight_layout()
plt.show()



In [ ]:

fig, axs = plt.subplots(1,2,figsize=(8*2,6), sharex=True, sharey=True)
x = 'umap1'
y = 'umap2'
hue = 'uSubclass'

include = ['Inhibitory']
include_by = 'uClass'

hue_order = np.sort(adata_rna.obs[adata_rna.obs[include_by].isin(include)][hue].unique())

for ax, adata_mod in zip(axs, [adata_mer_nr, adata_rna]):
    sns.scatterplot(data=adata_mod.obs[adata_mod.obs[include_by].isin(include)].sample(frac=1, replace=False), 
                    x=x, y=y, 
                    hue=hue, 
                    hue_order=hue_order,
                    # palette=clsts_palette3, hue_order=list(clsts_palette3.keys()),
                    s=5, edgecolor='none', ax=ax, rasterized=True)
    ax.axis('off')
    ax.set_aspect('equal')
    ax.legend(bbox_to_anchor=(1,1), fontsize=10)

axs[0].set_title('snRNA-seq (Cheng et al. 2022)')
axs[1].set_title('MERFISH (Xie et al. 2025)')
# powerplots.savefig_autodate(fig, os.path.join(outdir, 'fig1_umap_exc.pdf'))
fig.tight_layout()
plt.show()


In [ ]:
fig, axs = plt.subplots(1,2,figsize=(8*2,6), sharex=True, sharey=True)
x = 'umap1'
y = 'umap2'
hue = 'uSubclass'

include = ['Excitatory']
include_by = 'uClass'

hue_order = np.sort(adata_rna.obs[adata_rna.obs[include_by].isin(include)][hue].unique())

for ax, adata_mod in zip(axs, [adata_mer_nr, adata_rna]):
    sns.scatterplot(data=adata_mod.obs[adata_mod.obs[include_by].isin(include)].sample(frac=1, replace=False), 
                    x=x, y=y, 
                    hue=hue, 
                    hue_order=hue_order,
                    # palette=clsts_palette3, hue_order=list(clsts_palette3.keys()),
                    s=5, edgecolor='none', ax=ax, rasterized=True)
    ax.axis('off')
    ax.set_aspect('equal')
    ax.legend(bbox_to_anchor=(1,1), fontsize=10)

axs[0].set_title('snRNA-seq (Cheng et al. 2022)')
axs[1].set_title('MERFISH (Xie et al. 2025)')
# powerplots.savefig_autodate(fig, os.path.join(outdir, 'fig1_umap_exc.pdf'))
fig.tight_layout()
plt.show()

# Rfx4, Sdc4, Mertk in Astrocytes

In [ ]:
from sklearn.decomposition import PCA

adata_rna_astro = adata_rna[adata_rna.obs['uClass']=='Astrocytes']

mat = np.array(adata_rna_astro.layers['lognorm'].todense())
pcs = PCA(n_components=10).fit_transform(mat)
pcs.shape
# mat.shape

In [ ]:
gns = ['Rfx4', 'Sdc4', 'Mertk']
n = len(gns)
fig, axs = plt.subplots(1,n,figsize=(5*n,4))
for gn, ax in zip(gns, axs):
    gexp = np.array(adata_rna_astro[:,gn].layers['lognorm'].todense()).reshape(-1,)
    vmin = np.percentile(gexp, 0)
    vmax = np.percentile(gexp,95)
    g = ax.scatter(pcs[:,1], pcs[:,2], c=gexp, s=3, 
                cmap='rocket_r', vmin=vmin, vmax=vmax)
    ax.grid(False)
    sns.despine(ax=ax)
    ax.set_title(gn)
    fig.colorbar(g)

In [ ]:

gns = [
    'Gfap', 'Aqp4', 'Robo2', 
    'Trpm3','Grin2c', 'Mertk', 
    
    'Rfx4', 'Sdc4', 'Sema6d',
    'Cdh4',
]

n = len(gns)
nx = 3
ny = int((n+nx-1)/nx)

fig, axs = plt.subplots(ny,nx,figsize=(3.5*nx,3*ny), sharex=True, sharey=True)
for gn, ax in zip(gns, axs.flat):
    gexp = np.array(adata_rna_astro[:,gn].layers['lognorm'].todense()).reshape(-1,)
    vmin = np.percentile(gexp,  0)
    vmax = np.percentile(gexp, 99)
    g = ax.scatter(pcs[:,0], pcs[:,1], c=gexp, s=1, 
                cmap='coolwarm', vmin=vmin, vmax=vmax)
    ax.set_aspect('equal')
    ax.grid(False)
    sns.despine(ax=ax)
    ax.set_title(gn)
    fig.colorbar(g, shrink=0.3, aspect=5)
fig.tight_layout()

In [ ]:
# To store results under a specific name
np.random.seed(0)

res = 0.5
pcs_key = 'pc_astro'
neighbors_key = 'ngbr_astro'
clusters_key = 'cluster_res_p5'

adata_rna_astro.obsm[pcs_key] = pcs
sc.pp.neighbors(adata_rna_astro, n_neighbors=30, use_rep=pcs_key, key_added=neighbors_key)
sc.tl.leiden(adata_rna_astro, resolution=res, key_added=clusters_key, neighbors_key=neighbors_key)

dfplot = adata_rna_astro.obs
dfplot['pc1'] = pcs[:,0]
dfplot['pc2'] = pcs[:,1]

fig, ax = plt.subplots()
sns.scatterplot(data=dfplot, x='pc1', y='pc2', hue=clusters_key, s=10, ax=ax)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1,1))
plt.show()

In [ ]:
adata_rna_astro.obs['curated_type'] = adata_rna_astro.obs['cluster_res_p5']


In [ ]:
adata_rna_astro.obs.to_csv('/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/astro_cheng_subtypes_260128.csv')
# adata_rna_astro.write('/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/astro_cheng_subtypes_260128.h5ad')

In [ ]:

adata_data_raw = adata_merged_nr
adata_meta = adata_mer_nr

gns = [ 
       # 'Sema6a', 'Sema6d', 'Sdc4', 'Mertk', 
       # 'Plxna4', 'Plxna1', 'Col6a1', 'Gas6',
       # 'Cdh4', 'Negr1', 'Ptprm',
    'Gfap',
    'Trpm3', 
    'Mertk',
    'Sdc4',
    'Rfx4',
    'Megf10',
    
      ] 
n = len(gns)
x =  adata_meta.obs['width_show2']
y =  adata_meta.obs['depth_show2']

fig, axs = plt.subplots(n,1,figsize=(1*10,n*1))
for i, (ax, gn) in enumerate(zip(axs, gns)):
    # sample titles
    if i == 0:
        for lbl, coord in width_cum.items():
            ax.text(coord, 100, lbl, fontsize=8)# , pad=20)
    
    g = adata_data_raw[:,gn].layers['ljnorm'].reshape(-1,)
    vmax = np.percentile(g, 99)
    vmin = np.percentile(g,  0)
    sorting = np.argsort(g)
    
    p = utils_merfish.st_scatter_ax(fig, ax,  x[sorting],  y[sorting],  gexp=g[sorting], s=1, title='', vmin=vmin, vmax=vmax, cmap='rocket_r')
    # p = utils_merfish.st_scatter_ax(fig, ax,  x,  y,  gexp=g, s=1, title='', vmin=vmin, vmax=vmax, cmap='rocket_r')
    ax.set_title(gn, loc='left', va='center', ha='right', y=0.5, pad=None)
    fig.colorbar(p, pad=0, shrink=0.5, aspect=5, ticks=[np.round(vmin, decimals=1), np.round(vmax-0.1, decimals=1)])
    
fig_manager.savefig(fig)
plt.show()
    